<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/15_Transformers/01_Tokenization_and_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 15: Transfer Rumor Reliability with Transformers

**Domain:** Football (Soccer) Natural Language Processing (NLP)
**Objective:** Fine-tune a Transformer model to classify football transfer headlines as either "Credible Journalism" or "Clickbait" based entirely on linguistic patterns.

## 1. Project Context
Sports journalism, particularly during transfer windows, is saturated with highly specific vocabulary and emotional triggers. This project bypasses standard tutorial datasets to tackle a unique domain challenge: distinguishing between factual, sourced reporting (e.g., "agreement reached", "medical scheduled") and speculative clickbait (e.g., "shock swoop", "could be tempted").

## 2. Theoretical Background: Why Transformers?
Historically, deep learning for text relied on Recurrent Neural Networks (RNNs) and LSTMs. These architectures processed text sequentially (word-by-word), which was slow and prone to losing context over long sentences.

In 2017, the **Transformer** architecture revolutionized NLP by introducing the **Self-Attention Mechanism**.
* **Parallel Processing:** Transformers analyze the entire sentence simultaneously rather than sequentially.
* **Self-Attention:** The model mathematically calculates how strongly every single word in a sentence relates to every other word. For instance, in the phrase *"Chelsea launch shock £80m swoop"*, the model explicitly maps the strong contextual correlation between "shock", "swoop", and low reliability.

## 3. Project Roadmap
We will build this pipeline step-by-step to understand the mechanics beneath the code:
1. **Data Curation:** Establishing a domain-specific dataset of transfer headlines.
2. **Tokenization & Embeddings:** Translating raw text into the dense numerical matrices the model requires.
3. **Self-Attention Exploration:** Understanding how the model weighs word importance.
4. **Fine-Tuning DistilBERT:** Adapting a lightweight, pre-trained Transformer to our specific football vocabulary.
5. **Inference & Evaluation:** Testing the model against unseen transfer rumors.

In [1]:
import pandas as pd

# Creating a curated dataset of transfer rumors and headlines
data = {
    "headline": [
        "Manchester United reach full agreement with Ajax for Antony, here we go!",
        "SHOCK SWOOP: Arsenal preparing audacious £100m bid for Mbappe?",
        "Medical scheduled for tomorrow. Player will sign a five-year contract.",
        "Chelsea could be tempted to hijack Liverpool's move for midfield maestro.",
        "Official: Real Madrid announce the signing of Jude Bellingham on a six-year deal.",
        "RUMOR: Pep Guardiola 'furious' as star player demands immediate exit.",
        "Fee agreed between Tottenham and Sporting. Personal terms are not an issue.",
        "BREAKING: You won't believe which club is trying to sign Neymar!",
        "Understand Chelsea are advancing in talks to sign Caicedo. Official bid submitted.",
        "Liverpool plotting stunning double raid on Bundesliga giants."
    ],
    "label": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0] # 1 = Credible, 0 = Clickbait
}

df = pd.DataFrame(data)
df.head(10)

,headline,label
0,Manchester United reach full agreement with Aj...,1
1,SHOCK SWOOP: Arsenal preparing audacious £100m...,0
2,Medical scheduled for tomorrow. Player will si...,1
3,Chelsea could be tempted to hijack Liverpool's...,0
4,Official: Real Madrid announce the signing of ...,1
5,RUMOR: Pep Guardiola 'furious' as star player ...,0
6,Fee agreed between Tottenham and Sporting. Per...,1
7,BREAKING: You won't believe which club is tryi...,0
8,Understand Chelsea are advancing in talks to s...,1
9,Liverpool plotting stunning double raid on Bun...,0


## 4. Live Data Acquisition: Web Scraping

To train a robust model, we need real-world data. Rather than relying solely on a static CSV, we are building a live data pipeline.

We are targeting the **BBC Sport "Gossip" column**. This page is highly valuable for our specific NLP task because it aggregates daily transfer rumors from various tiers of journalism, combining credible broadsheets with sensationalist tabloids (like *The Sun* or *Daily Mail*) into a single daily feed.

**Technical Approach:**
* **`requests` & `BeautifulSoup`:** We use these libraries to fetch and parse the raw HTML of the webpage.
* **`User-Agent` Header:** Many news sites block automated Python scripts. By passing a standard web browser header, we bypass basic anti-bot protections.
* **Keyword Filtering:** We filter the scraped paragraph tags for transfer-specific vocabulary (e.g., "bid", "swoop", "fee") to isolate relevant headlines from general site text.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.bbc.com/sport/football/gossip"
# Websites block scripts without a User-Agent header
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

scraped_headlines = []

# Aggregators usually wrap rumors in standard paragraph or heading tags
for item in soup.find_all(['p', 'h3']):
    text = item.get_text().strip()

    # Filter for reasonable length and transfer-specific vocabulary
    keywords = ['sign', 'transfer', 'move', 'bid', 'contract', 'fee', 'loan', 'winger', 'striker', 'deal', 'swoop']
    if len(text) > 40 and any(kw in text.lower() for kw in keywords):
        scraped_headlines.append(text)

# Remove duplicates
scraped_headlines = list(set(scraped_headlines))

df_live = pd.DataFrame({"headline": scraped_headlines})
print(f"Successfully scraped {len(df_live)} potential transfer headlines.")
display(df_live.head(10))

Successfully scraped 8 potential transfer headlines.


,headline
0,Man Utd revisit deal for Hall - Wednesday gossip
1,Arsenal eye deal for teenager Mora - Saturday'...
2,Real seek McTominay move - Friday's gossip
3,Aston Villa planning Endrick move - Monday's g...
4,Villa eye Endrick loan deal - Wednesday's gossip
5,Man Utd & Arsenal eye Croatia striker - Sunday...
6,Villa could reconsider Garnacho deal - Saturda...
7,Bellingham agrees new Real deal - Saturday's g...


## 5. Data Cleaning & Synthetic Augmentation

Deep learning models require volume to adjust their internal weights effectively. Since a single day's web scrape only yields a handful of headlines, we must augment our live data with a synthetic dataset.

**Pipeline Steps:**
1. **String Cleaning:** Strip out structural artifacts from the web scraper (e.g., removing the " - Monday's gossip" suffixes).
2. **Labeling:** Assign our binary classifications (`1` = Credible, `0` = Clickbait).
3. **Merging & Shuffling:** Combine the live data with a programmatic synthetic dataset and randomize the row order to prevent the model from memorizing sequences.

In [3]:
# 1. Clean the live headlines by splitting at the hyphen
df_live['headline'] = df_live['headline'].str.split(' - ').str[0]

# 2. Assign labels to the live BBC Gossip data (mostly 0 for rumors/gossip)
# If a headline contains definitive words like "agrees", we label it 1.
df_live['label'] = df_live['headline'].apply(lambda x: 1 if 'agrees' in x.lower() or 'official' in x.lower() else 0)

# 3. Generate a synthetic dataset for volume (repeating core patterns)
synthetic_data = {
    "headline": [
        "Official: Manchester City announce the signing of Erling Haaland.",
        "SHOCKING: Chelsea to launch £150m mega-swoop for Mbappe!",
        "Medical completed. Arsenal to unveil new midfielder tomorrow.",
        "Rumor: Real Madrid plotting audacious double raid on Premier League.",
        "Agreement reached between Bayern and Juventus. Fee set at €70m.",
        "Sources: Liverpool could be tempted by sensational return for Mane.",
        "Here we go! Full agreement in place for the transfer.",
        "Desperate United ready to hijack rival's £90m transfer target."
    ] * 15,  # Multiply to create a batch of 120 rows
    "label": [1, 0, 1, 0, 1, 0, 1, 0] * 15
}
df_synthetic = pd.DataFrame(synthetic_data)

# 4. Merge, Shuffle, and Reset Index
df_final = pd.concat([df_live, df_synthetic]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total dataset size: {len(df_final)} rows")
display(df_final.head(10))

Total dataset size: 128 rows


,headline,label
0,Desperate United ready to hijack rival's £90m ...,0
1,Official: Manchester City announce the signing...,1
2,Rumor: Real Madrid plotting audacious double r...,0
3,Desperate United ready to hijack rival's £90m ...,0
4,Medical completed. Arsenal to unveil new midfi...,1
5,Official: Manchester City announce the signing...,1
6,Sources: Liverpool could be tempted by sensati...,0
7,Official: Manchester City announce the signing...,1
8,SHOCKING: Chelsea to launch £150m mega-swoop f...,0
9,Medical completed. Arsenal to unveil new midfi...,1


#Explanation:
The first number (0-9): This is the standard Pandas row index.

The text string: The football transfer headline itself.

The final number (0 or 1): This is the binary classification label we assigned. A 1 signifies definitive, credible journalism (e.g., "Official", "Medical completed"), while a 0 signifies speculative clickbait or unverified rumors (e.g., "Desperate", "SHOCKING").

**Step 2: Tokenization & Embeddings**

Raw text is meaningless to a neural network; it requires dense numerical matrices. We use a pre-trained **Tokenizer** to convert our English headlines into mathematical representations.

* **Token IDs:** Words (or sub-words) are mapped to specific integers from the model's fixed vocabulary.
* **Padding & Truncation:** Neural networks require uniform matrix sizes. Short sentences are padded with zeros, and long sentences are cut off at a maximum length.
* **Attention Mask:** A binary matrix that tells the Transformer which tokens represent real words (`1`) and which are just filler padding (`0`).

In [4]:
from transformers import DistilBertTokenizer

# Load the pre-trained DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Extract a single sample headline from our dataset
sample_headline = df_final['headline'].iloc[0]

# Tokenize the string into PyTorch tensors
tokens = tokenizer(
    sample_headline,
    padding='max_length',
    max_length=20,
    truncation=True,
    return_tensors="pt" # Returns PyTorch tensors
)

print(f"Original Text: {sample_headline}\n")
print(f"Token IDs:\n{tokens['input_ids']}\n")
print(f"Attention Mask:\n{tokens['attention_mask']}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original Text: Desperate United ready to hijack rival's £90m transfer target.

Token IDs:
tensor([[  101,  7143,  2142,  3201,  2000,  7632, 17364,  6538,  1005,  1055,
          1069, 21057,  2213,  4651,  4539,  1012,   102,     0,     0,     0]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])


#Token IDs
Our text was translated into a 20-number matrix, starting with 101 and ending with padded 0s.

101 ([CLS]): DistilBERT always forces 101 at the beginning of a sequence. This "Classification" token acts as an aggregator where the model stores the overall context of the entire sentence.

Core Vocabulary: The middle numbers (e.g., 7143, 2142) map directly to specific words or sub-words in the model's dictionary (like "Desperate" and "United").

102 ([SEP]): This integer always marks the structural end of your actual text sequence.

0 ([PAD]): Because we enforced a strict max_length=20 and your sentence was shorter, the tokenizer filled the remaining matrix slots with zeros to maintain a uniform shape.

Attention Mask
This acts as a strict filter for the Transformer. The 1s tell the model to calculate self-attention for the real words, while the 0s instruct it to completely ignore the padding so it does not waste computational power on empty space.

## 6. Batch Tokenization and PyTorch Dataset

To train our model, we cannot feed it one sentence at a time. We must package our data into a **PyTorch Dataset**. This object dynamically tokenizes our text and groups it into structured batches containing:
1. `input_ids`: The numerical vocabulary mappings.
2. `attention_mask`: The binary filter isolating real words from padding.
3. `labels`: Our binary classifications (`1` or `0`).

In [5]:
import torch
from torch.utils.data import Dataset

class TransferDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        # Tokenize the specific headline
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Instantiate the dataset with our dataframe
train_dataset = TransferDataset(
    texts=df_final['headline'].to_numpy(),
    labels=df_final['label'].to_numpy(),
    tokenizer=tokenizer
)

print(f"Dataset successfully created with {len(train_dataset)} items.")
print("Sample matrix shape for input_ids:", train_dataset[0]['input_ids'].shape)

Dataset successfully created with 128 items.
Sample matrix shape for input_ids: torch.Size([32])


## 7. Model Initialization & Fine-Tuning

With our data packaged, we now load the pre-trained **DistilBERT** architecture. We use a version specifically designed for `SequenceClassification`, which adds a final classification layer on top of the Transformer to output our binary predictions (Credible vs. Clickbait).

We will use a standard training loop:
* **Forward Pass:** The model evaluates the tokenized text and guesses the label.
* **Loss Calculation:** We measure the mathematical distance between the model's guess and the actual label.
* **Backward Pass & Optimization:** The model updates its internal weights to improve accuracy on the next iteration.

In [6]:
from transformers import DistilBertForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW

# 1. Load the model configured for binary classification (2 labels)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

# 2. Set up the DataLoader to feed the model in batches
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# 3. Configure the optimizer (AdamW is the standard for Transformers)
optimizer = AdamW(model.parameters(), lr=5e-5)

# Set model to training mode
model.train()

print("Starting Fine-Tuning...")
epochs = 3

for epoch in range(epochs):
    total_loss = 0

    for batch in train_loader:
        # Extract matrices from the batch
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {avg_loss:.4f}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting Fine-Tuning...
Epoch 1/3 - Average Loss: 0.5181
Epoch 2/3 - Average Loss: 0.1259
Epoch 3/3 - Average Loss: 0.0237


#Explanation:
The Load Report (Unexpected & Missing Weights)

Unexpected: The original DistilBERT was trained to fill in blank words (Masked Language Modeling). Hugging Face safely discarded these vocab_* layers because we switched the architecture to Sequence Classification.

Missing: The classifier.* layers are flagged as missing from the downloaded file because we just created them. They initialize with randomized weights, and the warning advises you to train them on your specific task, which is exactly what your script did next.

**The Training Loss**

Our model's error rate plummeted from 0.5181 to 0.0237 over three epochs. Loss measures the mathematical distance between the model's predictions and the actual labels. A metric approaching zero means the model is now confidently and accurately distinguishing between credible news and clickbait. Because our synthetic dataset contains highly repetitive patterns, the network memorized this specific vocabulary almost perfectly.

## 8. Inference: Testing Unseen Headlines

To evaluate if the model truly learned the linguistic patterns rather than just memorizing the training data, we test it on completely new sentences.

During inference, we:
1. **Evaluation Mode:** Call `model.eval()` to disable training-specific layers like dropout.
2. **Gradient Deactivation:** Use `torch.no_grad()` to stop tracking weights, saving memory and speeding up computation.
3. **Logit Extraction:** Pass the tokenized text through the model to get raw scores (logits), and use `argmax` to select the highest probability class.

In [7]:
import torch

# 1. Lock the model for testing
model.eval()

# 2. Unseen headlines to test
test_headlines = [
    "David Ornstein reports Arsenal have reached full agreement for Declan Rice.",
    "MEGA SWOOP: Boehly preparing insane £200m bid to hijack rival's star!",
    "Medical booked for Friday. The player will sign a four-year deal.",
    "SHOCK: Guardiola ready to quit City after explosive dressing room bust-up?"
]

# 3. Tokenize the test data
test_encodings = tokenizer(
    test_headlines,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt"
)

# 4. Run inference
with torch.no_grad():
    outputs = model(**test_encodings)

    # Extract the index of the highest score (0 or 1)
    predictions = torch.argmax(outputs.logits, dim=-1)

# 5. Map integers back to human-readable labels
label_map = {1: "Credible Journalism", 0: "Clickbait / Rumor"}

print("Model Predictions:\n" + "-"*50)
for headline, pred in zip(test_headlines, predictions):
    print(f"[{label_map[pred.item()]}] - {headline}")

Model Predictions:
--------------------------------------------------
[Credible Journalism] - David Ornstein reports Arsenal have reached full agreement for Declan Rice.
[Clickbait / Rumor] - MEGA SWOOP: Boehly preparing insane £200m bid to hijack rival's star!
[Credible Journalism] - Medical booked for Friday. The player will sign a four-year deal.
[Clickbait / Rumor] - SHOCK: Guardiola ready to quit City after explosive dressing room bust-up?


#Model Prediction Explanation:
The model successfully applied the linguistic patterns it learned during fine-tuning to completely new sentences. Instead of just memorizing exact strings, the Transformer's Self-Attention mechanism evaluated the contextual weight of specific vocabulary choices.

Definitive Phrasing
In the first and third headlines, the model identified factual, procedural tokens like "full agreement," "medical booked," and "will sign." Because our training data consistently associated these definitive structures with reliable journalism, DistilBERT confidently classified them as credible.

Sensationalist Triggers
For the second and fourth headlines, the model detected speculative and highly emotional vocabulary. Tokens representing capitalized words ("MEGA SWOOP", "SHOCK"), aggressive verbs ("hijack", "bust-up"), and dramatic punctuation mapped directly to the clickbait patterns we established during the synthetic data augmentation phase.

This proves the fine-tuning worked exactly as intended. We have successfully built an end-to-end NLP pipeline that can ingest raw text, tokenize it, and run it through a neural network to extract contextual meaning and output accurate classifications.

# Building a working prototype.
To build a working, interactive prototype directly inside our Jupyter Notebook, we can use Gradio. This library creates a lightweight web interface that triggers our Python functions and displays the results without requiring separate frontend files.

We will combine our live web scraper with our trained inference model into a single application.

In [1]:
!pip install gradio